# GMSH Mesh Generation via MeepMeshSim

This notebook demonstrates using `MeepMeshSim` to generate a GMSH mesh
from any gdsfactory component + LayerStack — mirroring Palace's
`mesh()` → `write_config()` workflow for photonic simulations.

We use a UBC PDK Y-branch (photonic SOI) as an example.

### Load component from UBC PDK

In [ ]:
from ubcpdk import PDK, cells

PDK.activate()

c = cells.ebeam_y_1550()
c

### Configure MeepMeshSim

In [ ]:
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.meep import MeepMeshSim

# Build a LayerStack for the SOI process
stack = LayerStack(pdk_name="ubcpdk")

stack.layers["core"] = Layer(
    name="core",
    gds_layer=(1, 0),
    zmin=0.0,
    zmax=0.22,
    thickness=0.22,
    material="si",
    layer_type="dielectric",
)

stack.dielectrics = [
    {"name": "box", "material": "SiO2", "zmin": -3.0, "zmax": 0.0},
    {"name": "clad", "material": "SiO2", "zmin": 0.0, "zmax": 1.8},
]

stack.materials = {
    "si": {"type": "dielectric", "permittivity": 11.7},
    "SiO2": {"type": "dielectric", "permittivity": 2.1},
}

# Configure simulation
sim = MeepMeshSim()
sim.geometry(component=c, stack=stack)
sim.materials = {"si": 3.47, "SiO2": 1.44}
sim.source(port="o1", wavelength=1.55, wavelength_span=0.1, num_freqs=11)
sim.monitors = ["o1", "o2", "o3"]
sim.domain(pml=1.0, margin=0.5)
sim.solver(resolution=32)

### Generate mesh and write config

In [ ]:
result = sim.mesh("./mesh-ybranch", model_name="ybranch")
config_path = sim.write_config()

print(f"Mesh file: {result.mesh_path}")
print(f"File size: {result.mesh_path.stat().st_size / 1024:.0f} KB")
print(f"Config: {config_path}")

### Inspect mesh statistics

In [ ]:
import json

# Mesh statistics
stats = result.mesh_stats
print("=== Mesh Stats ===")
print(json.dumps(stats, indent=2))

# Config contents
print("\n=== mesh_config.json ===")
print(config_path.read_text())

### Inspect physical groups

In [ ]:
groups = result.groups

print("Volume groups (materials):")
for name, info in groups["volumes"].items():
    print(f"  {name}: phys_group={info['phys_group']}, tags={info['tags']}")

print("\nLayer volumes:")
for name, info in groups["layer_volumes"].items():
    print(f"  {name}: phys_group={info['phys_group']}, material={info['material']}, tags={info['tags']}")

if groups.get("outer_boundary"):
    print(f"\nOuter boundary: phys_group={groups['outer_boundary']['phys_group']}")
else:
    print("\nNo outer boundary (airbox not included)")

### Visualize the mesh

In [ ]:
# Full mesh wireframe
sim.plot_mesh(interactive=False)

In [ ]:
# Show only the waveguide core
sim.plot_mesh(show_groups=["core"], interactive=False)